In [0]:
%python
dbutils.widgets.text("environment", "dev")
env = dbutils.widgets.get("environment")
print(env)

In [0]:
USE IDENTIFIER(:environment || '_banking.gold')

In [0]:
-- CREATE TABLE IF NOT EXISTS dwh_batch_control (
--     run_id STRING COMMENT 'Pipeline run identifier',
--     target_table STRING COMMENT 'DWH target',
--     processing_date DATE COMMENT 'Business processing date',
--     watermark_from TIMESTAMP COMMENT 'Exclusive source lower bound',
--     watermark_to TIMESTAMP COMMENT 'Inclusive source upper bound',
--     records_read BIGINT COMMENT 'ODS rows read',
--     records_upserted BIGINT COMMENT 'Rows written',
--     status STRING COMMENT 'RUNNING, SUCCESS or FAILED',
--     start_time TIMESTAMP COMMENT 'Run start',
--     end_time TIMESTAMP COMMENT 'Run end',
--     error_message STRING COMMENT 'Failure detail'
--   ) USING DELTA
--   COMMENT 'DWH load audit; unpartitioned';


In [0]:
CREATE TABLE IF NOT EXISTS dim_date (
    DateSK INT COMMENT 'YYYYMMDD date key',
    CalendarDate DATE COMMENT 'Calendar date',
    DayOfMonth INT COMMENT 'Day number',
    MonthNumber INT COMMENT 'Month number',
    MonthName STRING COMMENT 'Month name',
    QuarterNumber INT COMMENT 'Quarter number',
    YearNumber INT COMMENT 'Calendar year',
    IsMonthEnd BOOLEAN COMMENT 'Month-end flag'
  ) USING DELTA
  COMMENT 'Calendar dimension; one row per date';

In [0]:
CREATE TABLE IF NOT EXISTS dim_customer (
    CustomerSK STRING COMMENT 'SHA-256(CustomerID|StartDate)',
    CustomerID STRING COMMENT 'Customer natural key',
    FullName STRING COMMENT 'Customer name',
    AddressLine1 STRING COMMENT 'Masked customer address',
    City STRING COMMENT 'Customer city',
    State STRING COMMENT 'Customer state',
    PostalCode STRING COMMENT 'Customer postal code',
    CustomerSegment STRING COMMENT 'Customer segment',
    KYCStatus STRING COMMENT 'KYC status',
    StartDate DATE COMMENT 'Inclusive validity start',
    EndDate DATE COMMENT 'Exclusive validity end',
    IsCurrent BOOLEAN COMMENT 'Current SCD-2 row',
    SourceIngestedAt TIMESTAMP COMMENT 'ODS lineage timestamp'
  ) USING DELTA
  COMMENT 'Customer SCD Type 2'
  cluster by (CustomerID, StartDate);

In [0]:
CREATE TABLE IF NOT EXISTS dim_branch (
    BranchSK STRING COMMENT 'SHA-256(BranchID)',
    BranchID STRING COMMENT 'Branch natural key',
    BranchName STRING COMMENT 'Branch name',
    City STRING COMMENT 'Branch city',
    State STRING COMMENT 'Branch state',
    Region STRING COMMENT 'Branch region',
    SourceIngestedAt TIMESTAMP COMMENT 'ODS lineage timestamp'
  ) USING DELTA
  COMMENT 'Branch SCD Type 1';

In [0]:
CREATE TABLE IF NOT EXISTS dim_product (
    ProductSK STRING COMMENT 'SHA-256(ProductID)',
    ProductID STRING COMMENT 'Product natural key',
    ProductName STRING COMMENT 'Product name',
    ProductCategory STRING COMMENT 'Product category',
    AnnualFee DECIMAL(18, 2) COMMENT 'Annual fee',
    SourceIngestedAt TIMESTAMP COMMENT 'ODS lineage timestamp'
  ) USING DELTA
  COMMENT 'Product SCD Type 1';

In [0]:
CREATE TABLE IF NOT EXISTS dim_account (
    AccountSK STRING COMMENT 'Account version hash',
    AccountID STRING,
    PrimaryCustomerID STRING,
    BranchID STRING,
    AccountType STRING,
    AccountStatus STRING,
    OpenDate DATE,
    CloseDate DATE,
    CurrencyCode STRING,
    StartDate DATE,
    EndDate DATE,
    IsCurrent BOOLEAN,
    SourceIngestedAt TIMESTAMP
  ) USING DELTA
  COMMENT 'dim_account banking DWH table; initially unpartitioned'

In [0]:

CREATE TABLE IF NOT EXISTS fact_transactions (
    DateSK INT COMMENT 'Transaction date key',
    TxnID STRING COMMENT 'Transaction degenerate key',
    AccountID STRING COMMENT 'Account degenerate key',
    CustomerSK STRING COMMENT 'Customer version at event date',
    BranchSK STRING COMMENT 'Servicing branch key',
    TransactionTimestamp TIMESTAMP COMMENT 'Event timestamp',
    TransactionType STRING COMMENT 'Deposit, withdrawal or other type',
    TransactionAmount DECIMAL(18, 2) COMMENT 'Signed transaction amount',
    CurrencyCode STRING COMMENT 'Currency',
    MerchantName STRING COMMENT 'Merchant',
    MerchantCategory STRING COMMENT 'Merchant category',
    TransactionStatus STRING COMMENT 'Posted or reversed',
    IsLateArrival BOOLEAN COMMENT 'Arrived after event date',
    SourceIngestedAt TIMESTAMP COMMENT 'ODS lineage timestamp'
  ) USING DELTA
  COMMENT 'Atomic transaction fact; initially unpartitioned; cluster by DateSK, BranchSK, CustomerSK';

CREATE TABLE IF NOT EXISTS fact_account_balance (
    DateSK INT COMMENT 'Balance date key',
    BalanceDate DATE COMMENT 'Business balance date',
    AccountID STRING COMMENT 'Account degenerate key',
    CustomerSK STRING COMMENT 'Primary customer version at balance date',
    BranchSK STRING COMMENT 'Servicing branch key',
    CurrencyCode STRING COMMENT 'Currency',
    AvailableBalanceAmount DECIMAL(18, 2) COMMENT 'Available balance',
    LedgerBalanceAmount DECIMAL(18, 2) COMMENT 'Ledger balance',
    OverdraftLimitAmount DECIMAL(18, 2) COMMENT 'Overdraft limit',
    SourceIngestedAt TIMESTAMP COMMENT 'ODS lineage timestamp'
  ) USING DELTA
  COMMENT 'Account-day periodic snapshot; initially unpartitioned; cluster by DateSK, AccountID';

CREATE TABLE IF NOT EXISTS bridge_account_customer (
    BridgeSK STRING COMMENT 'SHA-256(AccountID|CustomerID|StartDate)',
    AccountID STRING COMMENT 'Account key',
    CustomerID STRING COMMENT 'Owner natural key',
    CustomerSK STRING COMMENT 'Customer version at ownership start',
    OwnershipRole STRING COMMENT 'Primary or joint role',
    OwnershipPercent DECIMAL(9, 4) COMMENT 'Allocation percentage',
    StartDate DATE COMMENT 'Inclusive validity start',
    EndDate DATE COMMENT 'Exclusive validity end',
    IsCurrent BOOLEAN COMMENT 'Current ownership version',
    SourceIngestedAt TIMESTAMP COMMENT 'ODS lineage timestamp'
  ) USING DELTA
  COMMENT 'Effective-dated ownership bridge; unpartitioned; cluster by AccountID, CustomerSK';

CREATE TABLE IF NOT EXISTS bridge_transaction_product (
    TxnID STRING COMMENT 'Transaction key',
    LineNumber INT COMMENT 'Transaction line sequence',
    ProductSK STRING COMMENT 'Product dimension key',
    ProductID STRING COMMENT 'Product natural key',
    Quantity DECIMAL(18, 4) COMMENT 'Quantity',
    LineAmount DECIMAL(18, 2) COMMENT 'Line amount',
    SourceIngestedAt TIMESTAMP COMMENT 'ODS lineage timestamp'
  ) USING DELTA
  COMMENT 'Transaction-product line bridge; unpartitioned; cluster by TxnID, ProductSK';